# 1. Install necessary Libraries

In this part, we'll begin by installing necessary libraries needed for running our computer vision training and testing scripts

In [3]:
%pip install torch torchvision torchaudio
%pip install transformers scikit-learn pillow pandas numpy huggingface_hub ipywidgets opencv-python

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Obtaining dependency information for ipywidgets from https://files.pythonhosted.org/packages/56/6d/0d9848617b9f753b87f214f1c682592f7ca42de085f564352f10f0843026/ipywidgets-8.1.8-py3-none-any.whl.metadata
  Obtaining dependency information for widgetsnbextension~=4.0.14 from https://files.pythonhosted.org/packages/3f/0e/fa3b193432cfc60c93b42f3be03365f5f909d2b3ea410295cf36df739e31/widgetsnbextension-4.0.15-py3-none-any.whl.metadata
  Obtaining dependency information for jupyterlab_widgets~=3.0.15 from https://files.pythonhosted.org/packages/ab/b5/36c712098e6191d1b4e349304ef73a8d06aed77e56ceaac8c0a306c7bda1/jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/139.8 kB ? eta -:--:--
   -- ------------------------------------- 10.2/139.8 kB ? eta -:--:--
   -------- ------------------------------ 30.7/139.8 kB 435.7 kB/s eta 0:00:01
   ----------------- --------------------- 61.4/139.8 kB 465.5 kB/s eta 0:00:01
   -----------------------------


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Huggingface login via UI
This part is optional as you're still able to access huggingface API without authentication, just that there's stricter rate limiting. If you have not faced any rate limiting issues can comment out or delete this cell

In [4]:
from huggingface_hub import notebook_login
notebook_login()

# 3. CLAHE Data Preprocessing function
In this part, we'll write Contrast Limited Adaptive Histogram Equalization (CLAHE) preprocessing function logic. Since this data preprocessing is added to all dataset (training/val/test) and its not part of data augmentation, we'll separate them for now

In [ ]:
import cv2
import numpy as np
from PIL import Image

def apply_clahe(image):
    image = np.array(image)

    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)

    merged = cv2.merge((cl, a, b))
    enhanced = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

    return Image.fromarray(enhanced)

# 4. Custom Dataset Loader
Since we're using a custom dataset, we will need to write a custom dataset loader to pass our image data to the model


In [5]:
from torch.utils.data import Dataset
from PIL import Image
import os

class MedicalDatasetLoader(Dataset):
    def __init__(self, df, img_dir, transform=None, use_clahe=True):
        self.data = df # Differentiate between train_df, val_df and test_df
        self.img_dir = img_dir # Image Directory Path
        self.transform = transform # Data augmentation
        self.use_clahe = use_clahe # Apply CLAHE data preprocessing

    def __len__(self):
        return len(self.data) # Calculate the number of rows of the dataset

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['Image'] # Get the image name from the csv header. Used for path later
        label = int(self.data.iloc[idx]['Label']) # Get the Label value from csv header

        img_path = os.path.join(self.img_dir, img_name) # Combine the root image dir with the image name to get the full individual image path
        image = Image.open(img_path).convert("RGB")

        if self.use_clahe: # Implement CLAHE for training set, disable for val and testing set
            image = apply_clahe(image)

        if self.transform: # Implement Data Augmentation
            image = self.transform(image)

        return image, label

# 5. Setup MPS Device
Here in this part, we check if PyTorch detects our GPU MPS device on macOS

In [6]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") # Check if MPS is detected by pytorch, fallback to CPU otherwise
print("Using device:", device)

Cuda available:  True


# 6. Setup Model
In this part, we will load the SwinV2 (Swin Transformer V2) model from HuggingFace. The model is pretrained on ImageNet-1K at 256x256 resolution but we use 224x224 for cross-model consistency.

In [ ]:
from transformers import Swinv2ForImageClassification, AutoImageProcessor

# Load Microsoft SwinV2 model
model = Swinv2ForImageClassification.from_pretrained(
    'microsoft/swinv2-base-patch4-window8-256',
    num_labels=5,
    ignore_mismatched_sizes=True
).to(device)

processor = AutoImageProcessor.from_pretrained('microsoft/swinv2-base-patch4-window8-256')

# 7. Data Preprocessing & K-Fold Setup
In this part, we add data augmentation to the training transforms and set up Stratified K-Fold cross-validation. We hold out 20% as a fixed test set, then use 5-fold StratifiedKFold on the remaining 80%. DataLoaders are created per-fold inside the training loop.

In [ ]:
from torchvision import transforms
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch.utils.data import DataLoader, WeightedRandomSampler
import pandas as pd

# Data augmentation for training (applied after CLAHE)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

# No augmentation for validation and testing
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

df = pd.read_csv('dataset/labels.csv')

# Step 1: Hold out 20% as fixed test set (never touched during K-Fold)
train_val_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['Label'],
    random_state=42
)

# Step 2: Setup 5-Fold Stratified K-Fold on the remaining 80%
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Create test dataset and loader (fixed across all folds)
test_dataset = MedicalDatasetLoader(test_df, "dataset/image/", val_transform, use_clahe=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Total samples: {len(df)}")
print(f"Train+Val pool: {len(train_val_df)} (will be split into {N_FOLDS} folds)")
print(f"Test (held out): {len(test_df)}")

# 8. Hyperparameters tuning
We set up our class weights, epochs number, early stopping mechanism, optimizer and learning rate scheduler to optimize our model performance and provide better generalization through reducing overfitting

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.utils.class_weight import compute_class_weight

class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha = alpha  # Per-class weights tensor on device
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        # Apply label smoothing via cross_entropy
        ce_loss = F.cross_entropy(
            logits, targets,
            weight=self.alpha,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )
        # Get predicted probabilities for focal modulation
        probs = F.softmax(logits, dim=1)
        p_t = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        # Apply focal modulation: (1 - p_t)^gamma
        focal_weight = (1 - p_t) ** self.gamma
        loss = focal_weight * ce_loss
        return loss.mean()

# Shared hyperparameter constants
NUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 16

# 9. Experiment Logging Class
In this part, we'll create a log class tha can help us to logs our hyperparameter lists as well as result metrics.

In [ ]:
import json
from datetime import datetime
import os

class ExperimentTracker:
    def __init__(self, base_dir="experiments"):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.exp_dir = os.path.join(base_dir, f"exp_{timestamp}")
        os.makedirs(self.exp_dir)

        self.epoch_metrics = []
        self.fold_metrics = []
        self.final_metrics = {}
        self.config = {}

    # ---------------- CONFIG ----------------
    def log_config(self, config):
        self.config = config
        with open(os.path.join(self.exp_dir, "config.json"), "w") as f:
            json.dump(config, f, indent=4)

    # ---------------- PER EPOCH ----------------
    def log_epoch(self, fold, epoch, train_loss, val_loss, train_acc, val_acc):
        self.epoch_metrics.append({
            "fold": fold,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc
        })

    # ---------------- PER FOLD ----------------
    def log_fold_metrics(self, fold, acc, prec, rec, f1, roc_auc):
        self.fold_metrics.append({
            "fold": fold,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1,
            "roc_auc_score": roc_auc
        })

    # ---------------- AGGREGATED K-FOLD ----------------
    def log_aggregated_metrics(self):
        if not self.fold_metrics:
            return {}
        metrics_keys = ["accuracy", "precision", "recall", "f1_score", "roc_auc_score"]
        aggregated = {}
        for key in metrics_keys:
            values = [fm[key] for fm in self.fold_metrics]
            aggregated[key] = {
                "mean": float(np.mean(values)),
                "std": float(np.std(values)),
                "per_fold": values
            }
        self.final_metrics["kfold_aggregated"] = aggregated
        return aggregated

    # ---------------- FINAL METRICS ----------------
    def log_final_metrics(self, split, acc, prec, rec, f1, roc_auc, cm):
        self.final_metrics[split] = {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1,
            "roc_auc_score": roc_auc,
            "confusion_matrix": cm.tolist()
        }

    # ---------------- SAVE ----------------
    def save_all(self):
        with open(os.path.join(self.exp_dir, "metrics.json"), "w") as f:
            json.dump({
                "config": self.config,
                "epoch_metrics": self.epoch_metrics,
                "fold_metrics": self.fold_metrics,
                "final_metrics": self.final_metrics
            }, f, indent=4)

    def save_model(self, model, name="best_model.pth"):
        torch.save(model.state_dict(), os.path.join(self.exp_dir, name))

# 10. Training and Validation Script
In this part, we train and test our model and finally save the final weights of the model

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

CONFIG = {
    "model": "microsoft/swinv2-base-patch4-window8-256",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "scheduler": "CosineAnnealingLR",
    "loss": "FocalLoss",
    "focal_gamma": 2.0,
    "label_smoothing": 0.1,
    "k_folds": N_FOLDS,
    "augmentation": "HFlip+VFlip+Rotation15+ColorJitter",
    "sampler": "WeightedRandomSampler"
}

tracker = ExperimentTracker()
tracker.log_config(CONFIG)

# Track best model across all folds
global_best_val_loss = float('inf')
best_fold = -1

for fold, (train_idx, val_idx) in enumerate(skf.split(train_val_df, train_val_df['Label'])):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    print(f"{'='*60}")

    # --- Split data for this fold ---
    fold_train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

    # --- Create datasets ---
    fold_train_dataset = MedicalDatasetLoader(fold_train_df, "dataset/image/", train_transform, use_clahe=True)
    fold_val_dataset = MedicalDatasetLoader(fold_val_df, "dataset/image/", val_transform, use_clahe=False)

    # --- WeightedRandomSampler for this fold's training data ---
    fold_labels = fold_train_df['Label'].values
    class_counts = np.bincount(fold_labels)
    sample_weights = 1.0 / class_counts[fold_labels]
    sample_weights = torch.DoubleTensor(sample_weights)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    # --- DataLoaders ---
    fold_train_loader = DataLoader(fold_train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
    fold_val_loader = DataLoader(fold_val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # --- Re-initialize model fresh for each fold ---
    model = Swinv2ForImageClassification.from_pretrained(
        'microsoft/swinv2-base-patch4-window8-256',
        num_labels=5,
        ignore_mismatched_sizes=True
    ).to(device)

    # --- Compute class weights for this fold ---
    class_weights = compute_class_weight('balanced', classes=np.unique(fold_train_df['Label']), y=fold_train_df['Label'])
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    # --- Loss, optimizer, scheduler ---
    criterion = FocalLoss(alpha=class_weights, gamma=2.0, label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    # --- Early stopping state ---
    best_val_loss = float('inf')
    epochs_no_improvement = 0

    for epoch in range(NUM_EPOCHS):
        # ---------------- TRAINING ----------------
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0

        for inputs, labels in fold_train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(pixel_values=inputs)
            loss = criterion(outputs.logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.logits, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # ---------------- VALIDATION ----------------
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_preds, all_labels, all_probs = [], [], []

        with torch.no_grad():
            for inputs, labels in fold_val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(pixel_values=inputs)
                loss = criterion(outputs.logits, labels)

                probs = F.softmax(outputs.logits, dim=1)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.logits, 1)

                val_total += inputs.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.detach().cpu().numpy())

        # ---------------- EPOCH METRICS ----------------
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total

        scheduler.step()

        tracker.log_epoch(fold + 1, epoch, epoch_train_loss, epoch_val_loss, epoch_train_acc, epoch_val_acc)

        # --- Early stopping ---
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            epochs_no_improvement = 0
            # Save if this is the best model across ALL folds
            if epoch_val_loss < global_best_val_loss:
                global_best_val_loss = epoch_val_loss
                best_fold = fold + 1
                tracker.save_model(model)
        else:
            epochs_no_improvement += 1
            if epochs_no_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"  Early stopping at epoch {epoch + 1}")
                break

        print(f"  Epoch [{epoch+1}/{NUM_EPOCHS}] Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

    # --- End-of-fold validation metrics ---
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro')
    try:
        roc_auc = roc_auc_score(np.array(all_labels), np.array(all_probs), multi_class='ovr')
    except ValueError:
        roc_auc = 0.0

    tracker.log_fold_metrics(fold + 1, acc, prec, rec, f1, roc_auc)
    print(f"\n  Fold {fold+1} Val — Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {roc_auc:.4f}")

# --- Aggregated K-Fold results ---
aggregated = tracker.log_aggregated_metrics()
print(f"\n{'='*60}")
print(f"K-FOLD AGGREGATED RESULTS (best model from fold {best_fold})")
print(f"{'='*60}")
for metric, vals in aggregated.items():
    print(f"  {metric}: {vals['mean']*100:.2f}% (+/- {vals['std']*100:.2f}%)")

tracker.save_all()

# 11. Testing Script & Metrics Visualization
Lastly, in the final parts we'll measure our model performance based on the following metrics:
- Accuracy: Calculate the ratio of correctly predicted labels
- Precision: Calculate how many positive predictions are truly positive (Minimize false positives)
- Recall: Calculate how many postivive labels are predicted (Minimize false negatives)
- F1-Score: Determine the mean of precision and recall combined
- ROC-AUC Score: Determine how confident our model is at classifying the classes (0.5 - Random Guessing, ~1.0: Almost 100% accurate)
- Confusion Matrix: Determine the total True Positive, False Positives, True Negatives and False Negatives that our model made

In [10]:
def evaluate_and_log(model, test_loader, device, tracker):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(pixel_values=inputs)

            probs = F.softmax(outputs.logits, dim=1)
            _, predicted = torch.max(outputs.logits, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro')
    cm = confusion_matrix(all_labels, all_preds)

    try:
        roc_auc = roc_auc_score(
            np.array(all_labels),
            np.array(all_probs),
            multi_class='ovr'
        )
    except ValueError:
        roc_auc = 0.0

    # 🔥 LOG TEST
    tracker.log_final_metrics("test", acc, prec, rec, f1, roc_auc, cm)
    tracker.save_all()

    print("\n📊 TEST RESULTS")
    print(f"Accuracy: {acc * 100:.2f}%")
    print(f"Precision: {prec * 100:.2f}%")
    print(f"Recall: {rec * 100:.2f}%")
    print(f"F1 Score: {f1 * 100:.2f}%")
    print(f"ROC-AUC: {roc_auc * 100:.2f}%")
    print("Confusion Matrix:")
    print(cm)

    return acc, prec, rec, f1, roc_auc, cm

Accuracy: 82.09%
Precision: 70.14%
Recall: 67.99%
F1-Score: 68.97%
Confusion Matrix: 
[[437   9   2   0   0]
 [ 17  58  27   1   2]
 [ 11  24 215  16  11]
 [  0   1  21  15   7]
 [  0   1  19   8  86]]
